# Гибридная RAG-система для поиска объявлений

Для каждого запроса нужно выбрать не более 50 объявлений из корпуса. Порядок этих
объявлений не влияет на метрику - главное полнота.

Стек текущей версии:
- BM25
- символьный TF–IDF — для устойчивости к опечаткам
- E5 small и base, дообученные на парах "запрос-объявление" train.parquet
- поиск по всей России и отдельные варианты с учётом географии
- Reciprocal Rank Fusion (RRF) для объединения всех списков

Реранкер для списков E5 не использован, но, возможно, зря, ибо до RRF Recall был по всему пулу 0.98, а после слияния терялось 0.07. Я почему-то хотел максимально ускорить процесс поиска, дабы больше соответствовать реальному поиску Авито.

## Описание решения

### Какие данные и признаки использованы

Из запроса используются:
- search_query — основной текст запроса
- search_infm_params_text — текст выбранных пользователем фильтров
- search_location_id — локация поиска
- search_category — категория поиска

Из объявления используются:
- item_title_raw — заголовок
- item_description_raw — описание
- item_infm_params_text — параметры услуги
- item_category_id и item_rating — для мягкого учёта фильтров
- item_location_id, item_latitude, item_longitude — для географического поиска
- item_id — только для формирования ответа

Цена, число отзывов, запрет сообщений и скрытый телефон не использовались: в экспериментах они не давали надёжного сигнала для этапа кандидатогенерации.

### Какие алгоритмы применены

BM25 хорошо находит объявления с теми же словами, что и в запросе. Символьный TF–IDF дополняет его при опечатках и разных формах слова. E5 переводит запрос и объявление в числовые векторы и позволяет находить совпадения по смыслу, даже если слова отличаются. Используются multilingual-e5-small и multilingual-e5-base. Обе модели полностью дообучаются на положительных парах "запрос — объявление".

Каждый способ поиска создаёт собственный список. Есть глобальные списки, списки внутри точной локации, поиск в радиусах 25/50/100 км и поиск в соседних локациях, которые часто выбирали пользователи из train. Затем 46 списков объединяются методом RRF: объявление получает больше баллов, если оно находится высоко сразу в нескольких списках.

### Как проверялось качество

Во время экспериментов данные делились по нормализованному тексту запроса, чтобы одинаковый запрос не оказался одновременно в обучении и проверке. Параметры RRF подбирались на tune, а окончательное сравнение проводилось на отдельном holdout, который не использовался при подборе параметров.

### Какие ошибки были найдены

1. **Точные слова не всегда совпадают.** Например, формулировки «диагностика авто» и «проверка автомобиля» близки по смыслу, но различаются лексически. Для этого добавлен dense-поиск E5.
2. **В запросах встречаются опечатки и разные окончания.** Для них добавлены символьные n-граммы и стемминг.
3. **Глобально похожее объявление может находиться далеко от пользователя.** Поэтому добавлены точная локация, расстояние, радиусы и переходы между соседними локациями. Географические признаки дали один из самых больших приростов.
4. **Фильтр может быть полезен, но жёсткое ограничение иногда удаляет правильный ответ.** Поэтому фильтры создают дополнительные списки, а не заменяют основной поиск.
5. **В длинном описании много шума.** E5 отдельно кодирует короткое представление объявления и представление с описанием; оба варианта участвуют в ансамбле.
6. **Другой выбранный объект того же запроса мог случайно стать отрицательным примером при обучении.** Такие ложные отрицательные примеры маскируются.

Параметры ниже зафиксированы по результатам экспериментов и на benchmark не подбираются.

### Эксперименты

1. Baseline: BM25 + символьный TF–IDF + предобученная E5-small + RRF.
Результат: **0.792605**.
2. Представления текста: сравнение короткого текста объявления, описания, запроса без фильтров и с фильтрами. Совместное использование оказалось лучше.
3. Упор на гео: добавление local-поиска, мягкого geo-score (Dense similarity уменьшается на небольшой штраф, зависящий от расстояния), радиусы 25/50/100 км и location transitions.
Результат вырос до **0.823320**.
4. Fine-tuning E5: дообучение E5-small на парах "запрос - объявление" с in-batch negatives и маскированием ложных negatives.
Результат: **0.844101**.
5. Дополнительные проверки: hard negatives и разбиение длинных описаний на части не дали устойчивого улучшения.
6. Финальный ансамбль: E5-small, обученная на 60k пар, + E5-base, обученная на 30k пар, вместе с BM25, TF–IDF и географическими каналами.
Итоговый результат: **0.852066**.


## 1. Установка библиотек


In [ ]:
%pip install -q "transformers==4.44.2" "tokenizers==0.19.1" "huggingface-hub>=0.34,<1" "safetensors>=0.5.3" "nltk==3.9.1"


## 2. Пути, библиотеки и параметры

In [ ]:
from pathlib import Path
from functools import lru_cache
import gc
import html
import os
import random
import re

import numpy as np
import pandas as pd
from IPython.display import FileLink, display
from nltk.stem.snowball import RussianStemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from tqdm.auto import tqdm
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer


DATA_DIR = Path('/kaggle/input/datasets/mishashalagin/avito-bootcamp-v2')
TRAIN_PATH = DATA_DIR / 'train.parquet'
QUERIES_PATH = DATA_DIR / 'benchmark_queries.parquet'
ITEMS_PATH = DATA_DIR / 'benchmark_items.parquet'
ANSWER_PATH = Path('/kaggle/working/answer.csv')

SEED = 20260920
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.set_num_threads(min(8, os.cpu_count() or 2))
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print('Устройство:', torch.cuda.get_device_name(0))


MODELS = {
    'small_60k': {
        'name': 'intfloat/multilingual-e5-small',
        'revision': '614241f622f53c4eeff9890bdc4f31cfecc418b3',
        'pairs': 60_000,
        'train_batch': 16,
        'gradient_accumulation': 4,
        'encode_batch': 64,
    },
    'base_30k': {
        'name': 'intfloat/multilingual-e5-base',
        'revision': 'd128750597153bb5987e10b1c3493a34e5a4502a',
        'pairs': 30_000,
        'train_batch': 8,
        'gradient_accumulation': 8,
        'encode_batch': 24,
    },
}

GLOBAL_K = 200
LOCAL_K = 150
GEO_K = 150
RADIUS_K = 120
TRANSITION_K = 150
MAX_LENGTH = 256
FINETUNE_MAX_LENGTH = 128
FINETUNE_LR = 2e-5

RRF = {
    'dense_weight': 0.25,
    'char_weight': 0.5,
    'local_weight': 2.5,
    'geo_weight': 0.5,
    'filter_weight': 0.5,
    'filter_dense_weight': 1.0,
    'radius_weight': 1.5,
    'transition_weight': 1.5,
    'rrf_k': 10.0,
}


## 3. Чтение и подготовка данных

Затем из train выбираются уникальные положительные пары для fine-tuning.
Дополнительно по истории выборов считаются частые переходы между локациями (это дало улучшение метрики в ходе эксперимента) и
географические центры локаций. Эти данные нужны для поиска рядом с пользователем.


In [ ]:
def clean_text(value):
    text = html.unescape(str(value)).replace('\n', ' ').replace('\t', ' ')
    text = re.sub(r'<[^>]+>', ' ', text).lower().replace('ё', 'е')
    return re.sub(r'\s+', ' ', text).strip()


train = pd.read_parquet(TRAIN_PATH)
queries = pd.read_parquet(QUERIES_PATH)
items = pd.read_parquet(ITEMS_PATH)

text_columns = [
    'search_query', 'search_infm_params_text',
    'item_title_raw', 'item_description_raw', 'item_infm_params_text',
]
number_columns = [
    'search_location_id', 'search_category', 'search_is_delivery_search',
    'item_price', 'item_rating', 'item_rating_reviews_count',
    'item_category_id', 'item_microcat_id', 'item_location_id',
    'item_latitude', 'item_longitude',
]

for frame in (train, queries, items):
    for column in text_columns:
        if column in frame:
            frame[column] = frame[column].fillna('').astype(str)
    for column in number_columns:
        if column in frame:
            frame[column] = pd.to_numeric(frame[column], errors='coerce')
    for column in ('search_location_id', 'search_category', 'search_is_delivery_search'):
        if column in frame:
            frame[column] = frame[column].fillna(-1).astype('int64')
    if 'item_id' in frame:
        frame['item_id'] = frame['item_id'].astype(str)

N_ITEMS = len(items)
N_QUERIES = len(queries)

# Удаление дупликатов
train['_query_group'] = train['search_query'].map(clean_text)
pair_pool = (train.sample(frac=1, random_state=SEED)
             .drop_duplicates('_query_group')
             .drop_duplicates('item_id')
             .copy())

pair_pool['_query_text'] = [
    clean_text(query) if not clean_text(filters)
    else clean_text(query) + '. фильтры: ' + clean_text(filters)
    for query, filters in zip(pair_pool['search_query'], pair_pool['search_infm_params_text'])
]
pair_pool['_item_text'] = (
    pair_pool['item_title_raw'].map(clean_text) + '. ' +
    pair_pool['item_infm_params_text'].map(clean_text).str[:500] + '. ' +
    pair_pool['item_description_raw'].map(clean_text).str[:1200]
)
pair_pool['_category'] = (pd.to_numeric(pair_pool['item_category_id'], errors='coerce')
                          .fillna(-1).astype(int))
pair_pool['_random_order'] = np.random.default_rng(SEED).random(len(pair_pool))
fine_tune_pool = pair_pool.head(60_000).copy()

# Все известные выбранные объявления запроса нужны, чтобы не считать их negatives
selected_queries = set(fine_tune_pool['_query_group'])
positive_items = (train[train['_query_group'].isin(selected_queries)]
                  .groupby('_query_group')['item_id']
                  .agg(lambda values: set(map(str, values)))
                  .to_dict())

# Где находилось выбранное объявление при поиске из другой локации
transitions = train[['search_location_id', 'item_location_id']].dropna().astype('int64')
transitions = transitions[transitions['search_location_id'] != transitions['item_location_id']]
transition_counts = (transitions.groupby(['search_location_id', 'item_location_id'])
                     .size().rename('count').reset_index()
                     .sort_values(['search_location_id', 'count', 'item_location_id'],
                                  ascending=[True, False, True]))
transition_top = (transition_counts.groupby('search_location_id').head(3)
                  .groupby('search_location_id')['item_location_id'].agg(list).to_dict())

# Центр локации — медианные координаты уникальных объявлений этой локации
coordinate_columns = ['item_id', 'item_location_id', 'item_latitude', 'item_longitude']
coordinate_data = pd.concat([
    items[coordinate_columns],
    train[coordinate_columns].drop_duplicates('item_id'),
], ignore_index=True).drop_duplicates('item_id', keep='first')

for column in ('item_location_id', 'item_latitude', 'item_longitude'):
    coordinate_data[column] = pd.to_numeric(coordinate_data[column], errors='coerce')

valid_coordinates = (
    coordinate_data['item_location_id'].notna() &
    coordinate_data['item_latitude'].notna() &
    coordinate_data['item_longitude'].notna() &
    coordinate_data['item_latitude'].abs().le(90) &
    coordinate_data['item_longitude'].abs().le(180) &
    ((coordinate_data['item_latitude'] != 0) | (coordinate_data['item_longitude'] != 0))
)
location_centers = (coordinate_data.loc[valid_coordinates]
                    .groupby('item_location_id')[['item_latitude', 'item_longitude']]
                    .median()
                    .rename(columns={'item_latitude': 'lat', 'item_longitude': 'lon'})
                    .to_dict('index'))
location_centers = {int(location): center for location, center in location_centers.items()}

print(f'Train: {len(train):,} строк')
print(f'Benchmark: {N_QUERIES:,} запросов и {N_ITEMS:,} объявлений')
print('Пары для обучения:', {'small': 60_000, 'base': 30_000})


## 4. Текстовые представления и географические списки

Здесь готовятся два текста объявления для E5:

- short: заголовок и тип услуги — короткий и обычно самый точный текст
- description: заголовок и описание — больше контекста, но больше шума (но в E5 влазит только 256 токенов)

Также объявления группируются по локациям и заранее вычисляются списки объектов в радиусах 25, 50 и 100 км от каждой локации benchmark-запросов.


In [ ]:
items['_title'] = items['item_title_raw'].map(clean_text)
items['_params'] = items['item_infm_params_text'].map(clean_text)
items['_description'] = items['item_description_raw'].map(clean_text)

field_names = [
    'вид услуги', 'тип услуги', 'место оказания услуг', 'тип стоимости',
    'начальная цена', 'цена', 'опыт работы', 'пол', 'возраст', 'график работы',
    'онлайн-запись', 'выезд к клиенту', 'выезд', 'район', 'адрес',
    'специализация', 'формат занятий',
]
field_boundary = '|'.join(re.escape(name) for name in sorted(field_names, key=len, reverse=True))


def extract_service_fields(text):
    result = []
    for label in ('вид услуги', 'тип услуги'):
        match = re.search(
            r'\b' + re.escape(label) + r'\s*(.*?)(?=\b(?:' + field_boundary + r')\b|$)',
            text,
        )
        if match and match.group(1).strip():
            result.append(label + ' ' + match.group(1).strip()[:160])
    return '. '.join(result)


items['_service'] = items['_params'].map(extract_service_fields)
document_texts = {
    'short': (items['_title'] + '. ' + items['_service']).tolist(),
    'description': (items['_title'] + '. ' + items['_description'].str[:3500]).tolist(),
}


def numeric_column(frame, name, default=np.nan):
    if name not in frame:
        return np.full(len(frame), default, dtype=np.float32)
    return pd.to_numeric(frame[name], errors='coerce').to_numpy(dtype=np.float32)


item_locations = (pd.to_numeric(items['item_location_id'], errors='coerce')
                  .fillna(-1).to_numpy(dtype=np.int64))
item_categories = numeric_column(items, 'item_category_id', -1)
item_latitudes = numeric_column(items, 'item_latitude')
item_longitudes = numeric_column(items, 'item_longitude')
item_ratings = numeric_column(items, 'item_rating')

valid_item_coordinates = (
    np.isfinite(item_latitudes) & np.isfinite(item_longitudes) &
    (np.abs(item_latitudes) <= 90) & (np.abs(item_longitudes) <= 180) &
    ((item_latitudes != 0) | (item_longitudes != 0))
)

location_rows = {
    int(location): np.asarray(rows, dtype=np.int32)
    for location, rows in pd.Series(item_locations).groupby(item_locations).groups.items()
}

transition_rows = {}
for source_location, target_locations in transition_top.items():
    groups = [location_rows.get(int(location), np.empty(0, dtype=np.int32))
              for location in target_locations]
    groups = [group for group in groups if len(group)]
    transition_rows[int(source_location)] = (
        np.unique(np.concatenate(groups)) if groups else np.empty(0, dtype=np.int32)
    )


def top_indices(scores, k, rows=None):
    # Возвращает k лучших объявлений с детерминированной обработкой равенств
    if rows is None:
        rows = np.arange(len(scores), dtype=np.int32)
    else:
        rows = np.asarray(rows, dtype=np.int32)
    if len(rows) == 0:
        return rows

    values = scores[rows]
    finite = np.isfinite(values)
    rows, values = rows[finite], values[finite]
    k = min(k, len(rows))
    if k == 0:
        return np.empty(0, dtype=np.int32)

    if len(rows) > k:
        boundary = np.partition(values, len(values) - k)[len(values) - k]
        above = np.flatnonzero(values > boundary)
        ties = np.flatnonzero(values == boundary)
        ties = ties[np.argsort(rows[ties], kind='stable')[:k - len(above)]]
        selected = np.concatenate([above, ties])
        rows, values = rows[selected], values[selected]

    return rows[np.lexsort((rows, -values))]


def distances_from_location(location_id):
    center = location_centers.get(int(location_id))
    if center is None:
        return np.full(N_ITEMS, np.nan, dtype=np.float32)

    center_lat, center_lon = np.radians([center['lat'], center['lon']])
    latitudes = np.radians(item_latitudes)
    longitudes = np.radians(item_longitudes)
    haversine = (
        np.sin((latitudes - center_lat) / 2) ** 2 +
        np.cos(center_lat) * np.cos(latitudes) *
        np.sin((longitudes - center_lon) / 2) ** 2
    )
    distances = 6371.0 * 2 * np.arcsin(np.sqrt(np.clip(haversine, 0, 1)))
    distances[~valid_item_coordinates] = np.nan
    return distances.astype(np.float32)


geo_data = {}
for location_id in tqdm(sorted(set(queries['search_location_id'].astype(int))),
                        desc='Географические списки'):
    distances = distances_from_location(location_id)
    nearby = np.isfinite(distances) & (item_locations != location_id)
    geo_data[location_id] = {
        'penalty': (0.025 * np.log1p(np.nan_to_num(distances, nan=2000.0))).astype(np.float16),
        'r25': np.flatnonzero(nearby & (distances <= 25)).astype(np.int32),
        'r50': np.flatnonzero(nearby & (distances <= 50)).astype(np.int32),
        'r100': np.flatnonzero(nearby & (distances <= 100)).astype(np.int32),
    }


## 5. BM25 и символьный TF–IDF

В этой ячейке строятся два классических поисковых индекса.

BM25 работает со словами. Заголовок повторяется три раза, поэтому совпадение в заголовке важнее совпадения в длинном описании. Символьный TF–IDF работает с фрагментами по 3–5 символов и помогает при опечатках.

Для каждого benchmark-запроса сохраняются глобальные, локальные и учитывающие фильтры списки кандидатов.


In [ ]:
stemmer = RussianStemmer()


@lru_cache(maxsize=200_000)
def stem_word(word):
    return stemmer.stem(word) if re.search('[а-я]', word) else word


def tokenize(text):
    words = re.findall(r'[а-яa-z0-9]+', text.lower().replace('ё', 'е'))
    return [stem_word(word) for word in words]


word_documents = (
    items['_title'] + ' ' + items['_title'] + ' ' + items['_title'] + ' ' +
    items['_params'].str[:900] + ' ' + items['_description'].str[:4500]
)
word_vectorizer = CountVectorizer(
    tokenizer=tokenize,
    token_pattern=None,
    lowercase=False,
    min_df=1,
    max_features=250_000,
    dtype=np.float32,
)
word_counts = word_vectorizer.fit_transform(
    tqdm(word_documents, desc='Токенизация для BM25')
).tocsr()

document_lengths = np.asarray(word_counts.sum(axis=1)).ravel()
document_frequency = np.asarray((word_counts > 0).sum(axis=0)).ravel()
idf = np.log1p((N_ITEMS - document_frequency + 0.5) /
               (document_frequency + 0.5)).astype(np.float32)
k1, b = 1.5, 0.75
length_factor = k1 * (1 - b + b * document_lengths / max(document_lengths.mean(), 1))

for start in tqdm(range(0, N_ITEMS, 5000), desc='Веса BM25'):
    stop = min(start + 5000, N_ITEMS)
    data_start, data_stop = word_counts.indptr[start], word_counts.indptr[stop]
    denominator = np.repeat(length_factor[start:stop],
                            np.diff(word_counts.indptr[start:stop + 1]))
    frequencies = word_counts.data[data_start:data_stop]
    word_counts.data[data_start:data_stop] = (
        frequencies * (k1 + 1) / (frequencies + denominator) *
        idf[word_counts.indices[data_start:data_stop]]
    )

bm25_index = word_counts.T.tocsr()

char_vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    min_df=2,
    max_features=160_000,
    sublinear_tf=True,
    dtype=np.float32,
)
char_index = char_vectorizer.fit_transform(
    tqdm(items['_title'], desc='Символьный TF-IDF')
).T.tocsr()


def parse_query_filters(row):
    text = clean_text(row['search_infm_params_text'])
    rating_match = re.search(r'рейтинг[^0-9]*([1-5](?:[.,]\d+)?)', text)
    minimum_rating = float(rating_match.group(1).replace(',', '.')) if rating_match else None
    text = re.sub(r'рейтинг[^0-9]*[1-5](?:[.,]\d+)?\s*'
                  r'(?:звезды?|звезд)?\s*(?:и выше)?', ' ', text)
    text = re.sub(r'\b(?:вид услуги|тип услуги|тип стоимости)\b', ' ', text)
    return text.strip(), minimum_rating


lexical_names = [
    'bm25_global', 'char_global', 'bm25_local', 'char_local',
    'filtered_global', 'filtered_local',
]
lexical_lists = {
    name: np.full((N_QUERIES, 200), -1, dtype=np.int32)
    for name in lexical_names
}

for query_index, row in tqdm(
    enumerate(queries.to_dict('records')),
    total=N_QUERIES,
    desc='Лексический поиск',
):
    query_text = clean_text(row['search_query'])

    word_query = word_vectorizer.transform([query_text])
    word_query.data[:] = 1
    bm25_scores = (word_query @ bm25_index).toarray().ravel()
    char_scores = (char_vectorizer.transform([query_text]) @ char_index).toarray().ravel()

    local = location_rows.get(int(row['search_location_id']), np.empty(0, dtype=np.int32))
    result_lists = [
        top_indices(bm25_scores, GLOBAL_K, np.flatnonzero(bm25_scores > 0)),
        top_indices(char_scores, GLOBAL_K, np.flatnonzero(char_scores > 0)),
        top_indices(bm25_scores, LOCAL_K, local[bm25_scores[local] > 0]),
        top_indices(char_scores, LOCAL_K, local[char_scores[local] > 0]),
    ]

    filter_text, minimum_rating = parse_query_filters(row)
    category = int(row['search_category'])
    if filter_text or minimum_rating is not None or category > 0:
        filter_query = word_vectorizer.transform([filter_text])
        filter_query.data[:] = 1
        filter_scores = (filter_query @ bm25_index).toarray().ravel()
        combined_scores = bm25_scores + 0.35 * filter_scores

        allowed = bm25_scores > 0
        if category > 0:
            allowed &= item_categories == category
        if minimum_rating is not None:
            allowed &= np.isfinite(item_ratings) & (item_ratings >= minimum_rating)

        result_lists.extend([
            top_indices(combined_scores, GLOBAL_K, np.flatnonzero(allowed)),
            top_indices(combined_scores, LOCAL_K, local[allowed[local]]),
        ])
    else:
        result_lists.extend([np.empty(0, dtype=np.int32), np.empty(0, dtype=np.int32)])

    for name, indices in zip(lexical_names, result_lists):
        lexical_lists[name][query_index, :len(indices)] = indices

print('Размер BM25-индекса:', bm25_index.shape)
print('Размер char TF-IDF индекса:', char_index.shape)


## 6. Fine-tuning E5

В этой ячейке описано дообучение bi-encoder (было решено использовать именно такой тип вместо cross-encoder'а из-за скорости). Запрос и выбранное объявление кодируются одной E5-моделью. Их векторы должны стать близкими, а векторы запроса и остальных объявлений текущего batch - далёкими.

Если другое объявление из batch тоже известно как правильное для этого запроса, его оценка маскируется.

Функция возвращает уже дообученные модель и токенизатор. Все веса модели обновляются без заморозки весов.


In [ ]:
def mean_pooling(model, tokenized_batch):
    hidden_states = model(**tokenized_batch).last_hidden_state
    attention_mask = tokenized_batch['attention_mask'].unsqueeze(-1)
    vectors = (hidden_states * attention_mask).sum(1) / attention_mask.sum(1).clamp_min(1)
    return F.normalize(vectors, dim=1)


def fine_tune_e5(tag, settings, pairs):
    # Каждый вариант начинается с одного и того же seed
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    pairs = pairs.sort_values(['_category', '_random_order']).reset_index(drop=True)
    tokenizer = AutoTokenizer.from_pretrained(
        settings['name'], revision=settings['revision']
    )
    model = AutoModel.from_pretrained(
        settings['name'], revision=settings['revision']
    ).to(DEVICE).train()
    model.gradient_checkpointing_enable()

    optimizer = torch.optim.AdamW(model.parameters(), lr=FINETUNE_LR, weight_decay=0.01)
    scaler = torch.amp.GradScaler('cuda')
    optimizer.zero_grad(set_to_none=True)

    batch_size = settings['train_batch']
    accumulation = settings['gradient_accumulation']

    for step, start in enumerate(tqdm(range(0, len(pairs), batch_size),
                                      desc=f'Fine-tuning {tag}')):
        batch_frame = pairs.iloc[start:start + batch_size]
        if len(batch_frame) < 2:
            continue

        query_batch = tokenizer(
            ['query: ' + text for text in batch_frame['_query_text']],
            padding=True,
            truncation=True,
            max_length=FINETUNE_MAX_LENGTH,
            return_tensors='pt',
        )
        item_batch = tokenizer(
            ['passage: ' + text for text in batch_frame['_item_text']],
            padding=True,
            truncation=True,
            max_length=FINETUNE_MAX_LENGTH,
            return_tensors='pt',
        )
        query_batch = {name: tensor.to(DEVICE) for name, tensor in query_batch.items()}
        item_batch = {name: tensor.to(DEVICE) for name, tensor in item_batch.items()}

        with torch.amp.autocast('cuda', dtype=torch.float16):
            query_vectors = mean_pooling(model, query_batch)
            item_vectors = mean_pooling(model, item_batch)
            logits = (query_vectors @ item_vectors.T).float() / 0.05

            item_ids = batch_frame['item_id'].astype(str).tolist()
            query_groups = batch_frame['_query_group'].tolist()
            for query_position, query_group in enumerate(query_groups):
                known_positives = positive_items.get(query_group, set())
                for item_position, item_id in enumerate(item_ids):
                    if query_position != item_position and item_id in known_positives:
                        logits[query_position, item_position] = -1e4

            targets = torch.arange(len(batch_frame), device=DEVICE)
            loss = F.cross_entropy(logits, targets) / accumulation

        scaler.scale(loss).backward()

        last_batch = start + batch_size >= len(pairs)
        if (step + 1) % accumulation == 0 or last_batch:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

    model.eval()
    del optimizer, scaler
    gc.collect()
    torch.cuda.empty_cache()
    return model, tokenizer


## 7. Dense-поиск и географические каналы

Эта ячейка обучает модели по очереди и сразу использует каждую для поиска. Сначала кодируются benchmark-запросы, затем два представления всех объявлений. Сходство считается скалярным произведением нормализованных векторов, то есть эквивалентно cosine similarity.

Для каждой пары "запроса-объявление" создаются глобальные и географические списки. Эмбеддинги хранятся в FP16, чтобы уменьшить расход видеопамяти.


В текущем решении 46 каналов.

Лексические каналы:
- BM25 global
- char global
- BM25 local
- char local
- filtered global
- filtered local
= 6 каналов

Для одной E5-модели:
- два представления объявления: short и description
- для каждого plain-запроса: global, local, geo (Dense similarity уменьшается на небольшой штраф, зависящий от расстояния), r25, r50, r100, transition — 7
- для каждого filters-запроса: global, local, geo — 3

Получаем:
2 представления × (7 plain + 3 filters) = 20 каналов на модель

Две модели дают 40 dense/geo-каналов. Вместе с шестью лексическими получается 46.

In [ ]:
plain_queries = queries['search_query'].map(clean_text).tolist()
filter_values = queries['search_infm_params_text'].map(clean_text).tolist()
queries_with_filters = [
    query if not filters else query + '. фильтры: ' + filters
    for query, filters in zip(plain_queries, filter_values)
]

plain_kinds = ['global', 'local', 'geo', 'r25', 'r50', 'r100', 'transition']
filter_kinds = ['global', 'local', 'geo']


@torch.inference_mode()
def encode_texts(texts, prefix, model, tokenizer, batch_size, description):
    # Кодирует тексты пакетами и возвращает FP16-массив в исходном порядке
    texts = list(texts)
    order = np.argsort([len(text) for text in texts], kind='stable')
    result = np.empty((len(texts), model.config.hidden_size), dtype=np.float16)

    for start in tqdm(range(0, len(texts), batch_size), desc=description):
        indices = order[start:start + batch_size]
        batch = tokenizer(
            [prefix + texts[index] for index in indices],
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors='pt',
        )
        batch = {name: tensor.to(DEVICE) for name, tensor in batch.items()}
        vectors = mean_pooling(model, batch).float().cpu().numpy().astype(np.float16)
        result[indices] = vectors

    return result


def encode_unique_queries(texts, prefix, model, tokenizer, batch_size, description):
    codes, unique_texts = pd.factorize(np.asarray(texts, dtype=object), sort=False)
    unique_vectors = encode_texts(
        unique_texts.tolist(), prefix, model, tokenizer, batch_size, description
    )
    return unique_vectors[codes]


def retrieve_dense_lists(document_vectors, query_vectors):
    gpu_documents = torch.tensor(document_vectors, device=DEVICE, dtype=torch.float16)
    max_k = max(GLOBAL_K, LOCAL_K, GEO_K, RADIUS_K, TRANSITION_K)
    result = {}

    for mode, kinds in (('plain', plain_kinds), ('filters', filter_kinds)):
        for kind in kinds:
            result[f'{mode}_{kind}'] = np.full((N_QUERIES, max_k), -1, dtype=np.int32)

    query_rows = queries.to_dict('records')
    for start in tqdm(range(0, N_QUERIES, 32), desc='Dense retrieval'):
        stop = min(start + 32, N_QUERIES)
        plain_tensor = torch.tensor(
            query_vectors['plain'][start:stop], device=DEVICE, dtype=gpu_documents.dtype
        )
        filter_tensor = torch.tensor(
            query_vectors['filters'][start:stop], device=DEVICE, dtype=gpu_documents.dtype
        )
        score_blocks = {
            'plain': (plain_tensor @ gpu_documents.T).float().cpu().numpy(),
            'filters': (filter_tensor @ gpu_documents.T).float().cpu().numpy(),
        }

        for offset in range(stop - start):
            query_index = start + offset
            row = query_rows[query_index]
            location_id = int(row['search_location_id'])
            local = location_rows.get(location_id, np.empty(0, dtype=np.int32))
            geo = geo_data[location_id]
            transition = transition_rows.get(location_id, np.empty(0, dtype=np.int32))

            for mode, kinds in (('plain', plain_kinds), ('filters', filter_kinds)):
                scores = score_blocks[mode][offset]
                lists = [
                    top_indices(scores, GLOBAL_K),
                    top_indices(scores, LOCAL_K, local),
                    top_indices(scores - geo['penalty'], GEO_K),
                ]
                if mode == 'plain':
                    lists.extend([
                        top_indices(scores, RADIUS_K, geo['r25']),
                        top_indices(scores, RADIUS_K, geo['r50']),
                        top_indices(scores, RADIUS_K, geo['r100']),
                        top_indices(scores, TRANSITION_K, transition),
                    ])

                for kind, indices in zip(kinds, lists):
                    result[f'{mode}_{kind}'][query_index, :len(indices)] = indices

    del gpu_documents
    gc.collect()
    torch.cuda.empty_cache()
    return result


dense_sources = {}

for tag, settings in MODELS.items():
    training_pairs = fine_tune_pool.head(settings['pairs']).copy()
    model, tokenizer = fine_tune_e5(tag, settings, training_pairs)

    # После обучения модель переводится в FP16 и используется только для инференса
    model.half().eval()
    query_vectors = {
        'plain': encode_unique_queries(
            plain_queries, 'query: ', model, tokenizer, settings['encode_batch'],
            f'{tag}: запросы без фильтров',
        ),
        'filters': encode_unique_queries(
            queries_with_filters, 'query: ', model, tokenizer, settings['encode_batch'],
            f'{tag}: запросы с фильтрами',
        ),
    }

    for view in ('short', 'description'):
        document_vectors = encode_texts(
            document_texts[view], 'passage: ', model, tokenizer,
            settings['encode_batch'], f'{tag}: объявления / {view}',
        )
        view_sources = retrieve_dense_lists(document_vectors, query_vectors)
        for key, array in view_sources.items():
            mode, kind = key.split('_', 1)
            dense_sources[(tag, view, mode, kind)] = array
        del document_vectors, view_sources

    del model, tokenizer, query_vectors, training_pairs
    gc.collect()
    torch.cuda.empty_cache()

print('Dense-каналов:', len(dense_sources))


## 8. Объединение списков и создание answer.csv

Эта ячейка объединяет 6 лексических и 40 dense-списков с помощью RRF. Вес списка зависит от типа модели и географического ограничения.

После выбора 50 объявлений их внутренние номера заменяются на item_id.


In [ ]:
channels = []


def add_channel(name, values, weight):
    if weight > 0:
        channels.append({'name': name, 'values': values, 'weight': float(weight)})


# Шесть лексических каналов
add_channel('bm25_global', lexical_lists['bm25_global'], 1.0)
add_channel('char_global', lexical_lists['char_global'], RRF['char_weight'])
add_channel('bm25_local', lexical_lists['bm25_local'], RRF['local_weight'])
add_channel('char_local', lexical_lists['char_local'],
            RRF['char_weight'] * RRF['local_weight'])
add_channel('filtered_global', lexical_lists['filtered_global'], RRF['filter_weight'])
add_channel('filtered_local', lexical_lists['filtered_local'],
            RRF['filter_weight'] * RRF['local_weight'])

# По 20 dense-каналов от каждой E5-модели
for tag in MODELS:
    for view in ('short', 'description'):
        plain_weight = RRF['dense_weight'] / 2
        add_channel(f'{tag}_plain_{view}_global',
                    dense_sources[(tag, view, 'plain', 'global')], plain_weight)
        add_channel(f'{tag}_plain_{view}_local',
                    dense_sources[(tag, view, 'plain', 'local')],
                    plain_weight * RRF['local_weight'])
        add_channel(f'{tag}_plain_{view}_geo',
                    dense_sources[(tag, view, 'plain', 'geo')],
                    plain_weight * RRF['geo_weight'])

        for radius in ('r25', 'r50', 'r100'):
            add_channel(f'{tag}_plain_{view}_{radius}',
                        dense_sources[(tag, view, 'plain', radius)],
                        RRF['radius_weight'] / 6)

        add_channel(f'{tag}_plain_{view}_transition',
                    dense_sources[(tag, view, 'plain', 'transition')],
                    RRF['transition_weight'] / 2)

        filter_weight = RRF['filter_dense_weight'] / 2
        add_channel(f'{tag}_filters_{view}_global',
                    dense_sources[(tag, view, 'filters', 'global')], filter_weight)
        add_channel(f'{tag}_filters_{view}_local',
                    dense_sources[(tag, view, 'filters', 'local')],
                    filter_weight * RRF['local_weight'])
        add_channel(f'{tag}_filters_{view}_geo',
                    dense_sources[(tag, view, 'filters', 'geo')],
                    filter_weight * RRF['geo_weight'])


# Таблицы RRF позволяют быстро переводить место в списке в итоговый балл
maximum_rank = max(channel['values'].shape[1] for channel in channels)
rank_numbers = np.arange(maximum_rank + 1, dtype=np.float64)
rrf_tables = []
for channel in channels:
    table = channel['weight'] * RRF['rrf_k'] / (RRF['rrf_k'] + rank_numbers)
    table[0] = 0.0
    rrf_tables.append(table)

fallback_channel = [channel['name'] for channel in channels].index(
    'small_60k_plain_short_global'
)


def fuse_query_lists(query_lists):
    candidates = np.unique(np.concatenate(query_lists)).astype(np.int32)
    ranks = np.zeros((len(candidates), len(channels)), dtype=np.uint16)

    for channel_index, indices in enumerate(query_lists):
        positions = np.searchsorted(candidates, indices)
        ranks[positions, channel_index] = np.arange(1, len(indices) + 1, dtype=np.uint16)

    scores = np.zeros(len(candidates), dtype=np.float64)
    for channel_index, table in enumerate(rrf_tables):
        scores += table[ranks[:, channel_index]]

    # При полном равенстве используем порядок основной E5-small, затем индекс объекта
    fallback_rank = np.where(
        ranks[:, fallback_channel] > 0,
        ranks[:, fallback_channel],
        65535,
    )
    order = np.lexsort((candidates, fallback_rank, -scores))
    return candidates[order[:50]]


predictions = []
for query_index in tqdm(range(N_QUERIES), desc='RRF: итоговые 50 объявлений'):
    query_lists = []
    for channel in channels:
        values = channel['values'][query_index]
        query_lists.append(values[values >= 0])
    selected_rows = fuse_query_lists(query_lists)
    predictions.append(items['item_id'].iloc[selected_rows].tolist())

answer = pd.DataFrame({
    'query_id': queries['query_id'].astype(str),
    'answer': [' '.join(item_ids) for item_ids in predictions],
})
answer.to_csv(ANSWER_PATH, index=False, encoding='utf-8')


# Минимальная проверка формата перед отправкой
known_item_ids = set(items['item_id'])
assert list(answer.columns) == ['query_id', 'answer']
assert len(answer) == len(queries) and answer['query_id'].is_unique
for value in answer['answer']:
    item_ids = value.split()
    assert len(item_ids) == 50
    assert len(set(item_ids)) == 50
    assert set(item_ids) <= known_item_ids

print('Готово:', ANSWER_PATH)
print(f'{len(answer):,} запросов, по 50 уникальных объявлений')
display(answer.head(3))
display(FileLink(str(ANSWER_PATH)))
